# Bootstrap confidence intervals

Issue: #14

## Question

How do we tell a real gain from noise? We resample sessions to get an interval for one model, and an interval for the *difference* between two models.

## Data used

No OTTO data. Synthetic per-session scores with a known true difference.

Code: `notebooks/bootstrap.py` (import path `from bootstrap import bootstrap_ci, paired_bootstrap_diff`). It stays out of `src/ranking/` until issue #52.

In [ ]:
import sys
import time
from itertools import pairwise

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")
from bootstrap import bootstrap_ci, paired_bootstrap_diff

## Same seed, same answer

In [ ]:
rng = np.random.default_rng(0)
scores = rng.beta(2, 5, size=5000)
r1 = bootstrap_ci(scores, seed=42)
r2 = bootstrap_ci(scores, seed=42)
r3 = bootstrap_ci(scores, seed=43)
assert r1 == r2
assert r1 != r3
print("mean, low, high:", r1)

## Paired interval on data with a known true difference

Model A is better than B by 0.02 on average. Each session's difference has noise with standard deviation 0.10, so a few thousand sessions are needed to see it.

In [ ]:
def make_pair(n, true_diff, rng):
    a = rng.beta(2, 5, size=n)
    b = a - true_diff + rng.normal(0, 0.10, size=n)
    return a, b


rng = np.random.default_rng(1)
a, b = make_pair(20_000, 0.02, rng)
mean, low, high = paired_bootstrap_diff(a, b)
print(f"true diff 0.02 -> estimate {mean:.4f}, interval [{low:.4f}, {high:.4f}]")
assert low > 0, "the interval should exclude zero"

# With no true difference the interval should contain zero in about 95 percent of repeats.
rng = np.random.default_rng(2)
reps = 200
covered = 0
for i in range(reps):
    a, b = make_pair(1000, 0.0, rng)
    _, low, high = paired_bootstrap_diff(a, b, seed=i)
    covered += low <= 0 <= high
rate = covered / reps
print(f"true diff 0: interval contained zero in {rate:.1%} of {reps} repeats")
assert 0.90 <= rate <= 0.99

## Interval width shrinks with more sessions

In [ ]:
sizes = [100, 300, 1000, 3000, 10000]
widths = []
rng = np.random.default_rng(3)
for n in sizes:
    values = rng.beta(2, 5, size=n)
    _, low, high = bootstrap_ci(values, seed=0)
    widths.append(high - low)
assert all(w2 < w1 for w1, w2 in pairwise(widths)), widths

fig, ax = plt.subplots(figsize=(5, 3))
ax.loglog(sizes, widths, "o-")
ax.set_xlabel("sessions")
ax.set_ylabel("95% interval width")
ax.set_title("Width shrinks roughly like 1/sqrt(n)")
plt.show()
print(dict(zip(sizes, [round(w, 4) for w in widths])))

## Bad input is rejected

In [ ]:
for bad in ([1.0, 2.0, np.nan], [], np.ones((2, 2))):
    try:
        bootstrap_ci(bad)
    except ValueError as err:
        print("ValueError:", err)
    else:
        raise AssertionError("bad input was accepted")
try:
    paired_bootstrap_diff([1.0, 2.0], [1.0])
except ValueError as err:
    print("ValueError:", err)
else:
    raise AssertionError("different lengths were accepted")

## Speed at full evaluation size

In [ ]:
rng = np.random.default_rng(4)
big_a = rng.beta(2, 5, size=200_000)
big_b = big_a - 0.005 + rng.normal(0, 0.10, size=200_000)
start = time.time()
result = paired_bootstrap_diff(big_a, big_b, n_boot=1000)
elapsed = time.time() - start
print(f"200,000 sessions x 1,000 resamples: {elapsed:.1f} s, result {result}")
assert elapsed < 60

## Result

Same seed gives the same numbers. With a true gap of 0.02 the paired interval excludes zero. With no gap it contains zero about 95 percent of the time. Width shrinks as sessions grow. Bad input raises a clear error. 200,000 sessions with 1,000 resamples runs well under a minute.

Resampling is by session. If you resampled rows, the interval would be too narrow.

## What I learned

To be written by the owner of the project.